# Lab 2.3 · Household or district

Day 2 · Session 3 (Resample the right unit) · 25 min.

To save travel, Zulal's survey team did not phone households all over the region. It picked 6 districts and phoned 60 households in each. Jamila Al-Shuraim wants mean satisfaction with an interval. You bootstrap the 360 responses the obvious way, then the right way, and count which of the two keeps the 95% promise.

**AI policy.** AI-tutor: hints one step at a time, never the simulation.

**Reference state.** `data/survey.csv`, the 9,000 responses, and `U.two_stage_sample(gen, 6, 60)`, which draws the two-stage sample. Districts are chosen with probability proportional to their size, so the plain mean of the sample is a fair estimate. Nothing from another lab is needed.

**Tasks**
- [ ] 1 · Draw the two-stage sample and estimate (4')
- [ ] 2 · Bootstrap by household (5')
- [ ] 3 · Bootstrap by district (6')
- [ ] 4 · Coverage of both over 300 studies (6')
- [ ] 5 · Explain the difference in two sentences (4')
- [ ] ✱ Stretch · Redraw with 12 districts of 30 responses and compare the widths

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ds-201.git"
    if not Path("/content/ds-201").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ds-201"], check=True)
    os.chdir("/content/ds-201/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("zulal.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import zulal as U
np.set_printoptions(precision=3, suppress=True)
print("mode:", U.MODE)

In [ ]:
def percentile_interval(boot, level=0.95):
    a = (1 - level) / 2 * 100
    low, high = np.percentile(boot, [a, 100 - a])
    return float(low), float(high)


survey = U.survey()
TRUE = float(survey["satisfaction"].mean())
print(f"all 9,000 responses: mean satisfaction {TRUE:.3f} on the 10-point scale")

## 1 · Draw the two-stage sample and estimate (4')

In [ ]:
sample = U.two_stage_sample(U.rng(71), 6, 60)
est = float(sample["satisfaction"].mean())
by_district = sample.groupby("district")["satisfaction"].agg(["size", "mean"]).round(2)
print(by_district.to_string())
print(f"estimate from {len(sample)} responses in {sample['district'].nunique()} districts: {est:.3f}")

**Look at.** The six district means differ by more than a point. Two households of one district resemble each other more than two households picked anywhere: the rows are not 360 independent pieces of information.

## 2 · Bootstrap by household (5')

The obvious way: treat the 360 rows as the units.

✏️ **Exercise** · Write `boot_households(values, B, gen)`: `B` means of resamples of the rows.

In [ ]:
def boot_households(values, B, gen):
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
B = U.pick(2000, 500)
v = sample["satisfaction"].to_numpy(float)
try:
    bh = np.asarray(boot_households(v, B, U.rng(72)))
except NotImplementedError:
    bh = U.reference("d2_3", "bh")
U.keep("d2_3", "bh", bh)
ih = percentile_interval(bh)
print(f"household bootstrap: {ih[0]:.2f} to {ih[1]:.2f} (width {ih[1] - ih[0]:.2f})")
U.check(0.25 < ih[1] - ih[0] < 0.6, "resampling rows gives a narrow interval, about 0.4 points wide")

## 3 · Bootstrap by district (6')

The survey team drew **districts** first. So the unit to resample is the district: draw 6 districts with replacement from the 6, and take each drawn district with all its responses. With equal numbers per district, that is the mean of the resampled district means.

✏️ **Exercise** · Write `boot_districts(df, B, gen)`: compute the district means, resample them with replacement `B` times, return the `B` means.

In [ ]:
def boot_districts(df, B, gen):
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    bd = np.asarray(boot_districts(sample, B, U.rng(73)))
except NotImplementedError:
    bd = U.reference("d2_3", "bd")
U.keep("d2_3", "bd", bd)
idist = percentile_interval(bd)
print(f"district bootstrap:  {idist[0]:.2f} to {idist[1]:.2f} (width {idist[1] - idist[0]:.2f})")
print(f"the truth: {TRUE:.2f}; household interval {'contains' if ih[0] <= TRUE <= ih[1] else 'MISSES'} it, district interval {'contains' if idist[0] <= TRUE <= idist[1] else 'MISSES'} it")
U.check(idist[1] - idist[0] > 1.5 * (ih[1] - ih[0]), "resampling districts gives an interval about twice as wide")

## 4 · Coverage of both over 300 studies (6')

Which width is the honest one? Count.

✏️ **Exercise** · Write `coverage(studies, gen)`: for each study draw `U.two_stage_sample(gen, 6, 60)`, build both intervals with 1,000 resamples, and return the two shares that contain `TRUE`: `(household_share, district_share)`.

In [ ]:
def coverage(studies, gen):
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
STUDIES = U.pick(300, 80)
try:
    cov = coverage(STUDIES, U.rng(74))
except NotImplementedError:
    cov = U.reference("d2_3", "cov")
U.keep("d2_3", "cov", list(cov))
print(f"{STUDIES} two-stage studies: the household interval covers {cov[0]:.1%}, the district interval {cov[1]:.1%} (promised: 95%)")
U.check(cov[0] < 0.80 and cov[1] > 0.84, "the household bootstrap misses the truth in about a third of studies; the district bootstrap comes close to its promise")

**Look at.** The district interval still falls a little short of 95%: six districts are few, and a bootstrap of six values is rough. It is honest about the size of the uncertainty all the same; the household interval is not.

## 5 · Explain the difference in two sentences (4')

✏️ **Exercise** · Return your two sentences. The first names the unit that was sampled; the second says what resampling rows gets wrong.

In [ ]:
def my_explanation():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    text = my_explanation()
except NotImplementedError:
    text = U.reference("d2_3", "text")
U.keep("d2_3", "text", text)
print(text)
U.check("district" in text.lower() and len(text.split()) >= 20, "the explanation names the district as the sampled unit")

## ✱ Stretch · More districts, fewer calls in each

The same 360 calls spent on 12 districts of 30. Compare the width of the district interval.

In [ ]:
def stretch():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    U.keep("d2_3", "width_12", stretch())
except NotImplementedError:
    print("✱ stretch not done")

In [ ]:
U.note(f"Checked against the truth: coverage of the mean of all 9,000 responses ({TRUE:.2f}) over {STUDIES} two-stage studies: household {cov[0]:.0%}, district {cov[1]:.0%}.",
       "Not checked: a district bootstrap with unequal numbers of responses per district, which needs the full rows of each drawn district.")

## Checkpoint

1. A study measures 5 pressure readings on each of 40 pumps. What is the resampling unit for the mean pressure?
   <details><summary>Answer</summary>The pump: readings of one pump move together. Resample pumps, each with its 5 readings.</details>
2. The survey had phoned 360 households picked at random across the whole region. Which bootstrap then?
   <details><summary>Answer</summary>By household: the household was the unit drawn at random.</details>
3. In the capstone defence the assessor names another resampling unit. What do you predict before running it?
   <details><summary>Answer</summary>Whether the interval widens (a larger unit that groups similar rows) or stays about the same (rows inside the unit are no more alike than rows anywhere).</details>

## Carry forward

"What was the unit that was drawn at random?" is the first question of every interval and every test from here on.